# **Teknik Reduksi Dimensi (Dimensionality Reduction Techniques)**
Catatan ini mendalami teknik reduksi dimensi, yaitu proses memampatkan atau menyeleksi fitur data secara sistematis guna mengurangi beban komputasi tanpa mengorbankan integritas informasi fundamental. Pembahasan mencakup kerangka kerja Principal Component Analysis (PCA), Linear Discriminant Analysis (LDA), dan t-Distributed Stochastic Neighbor Embedding (t-SNE), serta evaluasi dampaknya terhadap keandalan klasifikasi.

Tujuan Pembelajaran :


*   Fundamental Reduksi Dimensi: Memahami urgensi dan implikasi kompresi fitur pada ekosistem algoritma.
*   Seleksi vs. Ekstraksi: Mampu membedakan paradigma Feature Selection dan Feature Extraction.







* PCA: Mengimplementasikan PCA dan mengartikan metrik variance, covariance, eigenvalue, serta eigenvector.

* LDA: Mengaplikasikan LDA untuk memaksimalkan separabilitas antar-kelas berbasis fungsi linear.

* t-SNE: Menggunakan t-SNE sebagai instrumen eksplorasi topologi lokal pada ruang berdimensi tinggi.

* Analisis Trade-Off: Mengevaluasi kompromi antara efisiensi komputasi, kejelasan interpretasi, dan ketepatan prediksi.

## **Bagian 1: Persiapan Lingkungan Kerja (Environment Setup)**
Modul ini beroperasi menggunakan pustaka analitik standar dari Python (numpy, pandas, matplotlib, scikit-learn) dan memanfaatkan dataset terintegrasi (Wine, Digits, dan Iris).

In [7]:
# Memuat pustaka manipulasi data dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# Menonaktifkan peringatan FutureWarning
warnings.simplefilter(action="ignore", category=FutureWarning)

# Memuat dataset referensi
from sklearn.datasets import load_wine, load_digits, load_iris

# Memuat pustaka evaluasi dan pemodelan
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans

# Memuat modul reduksi dimensi
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# Konfigurasi stabilitas dan visualisasi
np.random.seed(2024)
pd.set_option("display.max_columns", 100)

## **Bagian 2: Terminologi Ekstraksi vs. Seleksi Fitur**
Memiliki ratusan fitur sering kali memicu fenomena "Kutukan Dimensi" (Curse of Dimensionality), di mana model mengalami kelambatan komputasi, kesulitan interpretasi, dan overfitting. Terdapat dua jalan utama untuk mengatasi hal ini:

## **Bagian 3: Urgensi Penskalaan (Standardization)**
Sebelum algoritma reduksi dieksekusi, normalisasi magnitudo melalui standardisasi menjadi syarat absolut. Tanpa standardisasi, fitur dengan unit ukuran yang besar akan secara keliru mendominasi perumusan proyektor komponen utama (Principal Components).Formula Penskalaan Z-Score:$$z = \frac{x - \mu}{\sigma}$$(Di mana $x$ adalah fitur asli, $\mu$ rata-rata, dan $\sigma$ standar deviasi).

## **Bagian 4: Principal Component Analysis (PCA)**

PCA merupakan teknik transformasi ortogonal non-terawasi (unsupervised) yang mereduksi dimensi data sembari meminimalkan hilangnya variansi (informasi). PCA tidak memperhitungkan presensi atau sebaran label target.Landasan Matematis Singkat:

1. Hitung Covariance
Matrix ($S$) untuk melihat pergerakan linier antar-fitur:$$S = \frac{1}{n-1} \sum_{i=1}^{n} (X_i - \bar{X})(X_i - \bar{X})^T$$

2. Rumuskan relasi Eigenvector ($v$) dan Eigenvalue ($\lambda$) dari matriks kovarian:$$Sv = \lambda v$$(Eigenvector bertindak sebagai arah komponen utama, sedangkan eigenvalue menunjukkan skala informasi yang berhasil diselamatkan).

In [8]:
# Persiapan Dataset Wine
wine = load_wine()
df_wine = pd.DataFrame(data=wine.data, columns=wine.feature_names)
target_wine = wine.target

# Konstruksi Pipeline Standar -> PCA 2D
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

# Proyeksi Data
X_pca = pca_pipeline.fit_transform(df_wine)

# Evaluasi Rasio Penjelasan Variansi (Explained Variance Ratio)
pca_estimator = pca_pipeline.named_steps["pca"]
print("Proporsi Variansi Tertahan:", pca_estimator.explained_variance_ratio_)
print("Total Variansi yang Terselamatkan:", pca_estimator.explained_variance_ratio_.sum())

Proporsi Variansi Tertahan: [0.36198848 0.1920749 ]
Total Variansi yang Terselamatkan: 0.5540633835693526


## **Bagian 5: Linear Discriminant Analysis (LDA)**
Berbeda halnya dengan PCA, LDA adalah teknik terawasi (supervised). Alih-alih mengejar sumbu variansi terbesar, LDA beroperasi secara aktif untuk mencari sumbu proyeksi linear yang memaksimalkan jarak pemisah antar-kelas (class separability) sekaligus memadatkan sebaran intra-kelas.Batas maksimal komponen pada LDA dipatok pada formula: $\min(\text{Jumlah Fitur}, \text{Jumlah Kelas} - 1)$.

In [9]:
# Konstruksi Pipeline Standar -> LDA 2D
lda_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis(n_components=2))
])

# LDA Membutuhkan input target kelas (y_wine) pada fase fit
X_lda = lda_pipeline.fit_transform(wine.data, wine.target)

# Visualisasi dan komparasi dapat dilakukan menggunakan scatter plot matplotlib

## **Bagian 6: t-Distributed Stochastic Neighbor Embedding (t-SNE)**
t-SNE dikhususkan secara eksklusif untuk peruntukan visualisasi (pemetaan dimensi tinggi menuju bidang datar 2D atau ruang 3D). Algoritma ini bersifat non-linear dan dirancang untuk mempertahankan topologi kedekatan lokal (local neighborhood structure) antar objek observasi.

Perhatian: t-SNE tidak diperuntukkan sebagai alat transformasi kompresi langsung untuk melatih model regresi/klasifikasi tahap lanjut (bukan preprocessing pipeline tool konvensional).

In [10]:
# Eksplorasi menggunakan dataset Digits (Tulisan Tangan)
digits = load_digits()

# Pembatasan sampel dan eksekusi Skala + t-SNE
n_subset = 800
X_digits = StandardScaler().fit_transform(digits.data[:n_subset])
y_digits = digits.target[:n_subset]

# Inisialisasi t-SNE
tsne = TSNE(n_components=2, random_state=2024, perplexity=30, learning_rate="auto", init="pca")
X_tsne = tsne.fit_transform(X_digits)

## **Bagian 7: Dilema dan Performa Reduksi (Trade-Off)**
Reduksi dimensi selalu memicu dilema (trade-off). Pemampatan matriks meringankan komputasi (speed) dan mengekang ancaman memori berlebihan (overfitting), namun pencabutan informasi (information loss) dapat menggerus level keakuratan absolut.

In [11]:
# Demonstrasi Komparasi Prediksi (Tanpa PCA vs PCA 2D) pada dataset Iris
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target)

# Baseline 1: Regresi Logistik Biasa
pipe_without_pca = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

# Baseline 2: Regresi Logistik Terkompresi
pipe_with_pca = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=2)), ("model", LogisticRegression(max_iter=1000))])

# Evaluasi
pipe_without_pca.fit(X_train, y_train)
pipe_with_pca.fit(X_train, y_train)

print("Akurasi Model Tanpa PCA (4 Fitur):", accuracy_score(y_test, pipe_without_pca.predict(X_test)))
print("Akurasi Model Dengan PCA (2 Fitur):", accuracy_score(y_test, pipe_with_pca.predict(X_test)))

Akurasi Model Tanpa PCA (4 Fitur): 0.9111111111111111
Akurasi Model Dengan PCA (2 Fitur): 0.8888888888888888


## **Kesimpulan Strategis**
Teknik transformasi matriks mana yang ideal dipengaruhi oleh tujuan akhir analisis:

1. Identifikasi Klaster Laten: Gunakan PCA.

2. Optimalisasi Akurasi Kelas: Gunakan LDA.

3. Eksplorasi Representasi Grafis: Gunakan t-SNE.

Implementasi optimal dari reduksi dimensi bukan semata bertujuan menghapus kolom data, melainkan memfokuskan atensi model matematis pada resonansi fundamental yang menyembunyikan pola sebenarnya.